In [8]:
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
# 1. Chargement des données (Utilise tes noms de fichiers sauvegardés)
X_train = pd.read_csv("../features/global_features/X_train.csv")
y_train = pd.read_csv("../features/global_features/y_train.csv")
X_test = pd.read_csv("../features/global_features/X_test.csv")

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd

# 1. Identifier les colonnes CLIP (en supposant qu'elles contiennent 'clip' dans leur nom)
# Si elles n'ont pas de nom spécifique, adapte l'indexation (ex: X.iloc[:, :512])
clip_cols = [c for c in X_train.iloc[:,23:535].columns]
other_cols = [c for c in X_train.columns if c not in clip_cols]

print(f"Nombre de features CLIP détectées : {len(clip_cols)}")
print(f"Nombre d'autres features : {len(other_cols)}")

# 2. Initialiser la PCA
# n_components=32 est un bon point de départ pour 512 dimensions sur 1600 vidéos
n_components = 32 
pca = PCA(n_components=n_components, random_state=42)

# 3. Fit & Transform sur le TRAIN
clip_pca_train = pca.fit_transform(X_train[clip_cols])

# 4. Transform uniquement sur le TEST (on n'utilise pas fit ici !)
clip_pca_test = pca.transform(X_test[clip_cols])

# 5. Conversion en DataFrame pour reconstruction
clip_pca_train_df = pd.DataFrame(
    clip_pca_train, 
    columns=[f'pca_clip_{i}' for i in range(n_components)],
    index=X_train.index
)
clip_pca_test_df = pd.DataFrame(
    clip_pca_test, 
    columns=[f'pca_clip_{i}' for i in range(n_components)],
    index=X_test.index
)

# 6. Assemblage final : Autres features + Composantes PCA
X_train = pd.concat([X_train[other_cols], clip_pca_train_df], axis=1)
X_test = pd.concat([X_test[other_cols], clip_pca_test_df], axis=1)

print(f"Nouvelle forme de X_train : {X_train.shape}")
print(f"Variance expliquée cumulée : {pca.explained_variance_ratio_.sum():.2%}")

Nombre de features CLIP détectées : 1
Nombre d'autres features : 595


ValueError: n_components=32 must be between 0 and min(n_samples, n_features)=1 with svd_solver='covariance_eigh'

In [9]:
# 2. Préparation
y_train = y_train.values.flatten()
test_ids = X_test['video_id']
X = X_train.drop(columns=['video_id'], errors='ignore')
X_test_final = X_test.drop(columns=['video_id'], errors='ignore')

# 3. Configuration du K-Fold
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Listes pour stocker les scores de validation et les prédictions finales
oof_preds = np.zeros(len(X)) # Out-of-fold predictions
test_preds = np.zeros(len(X_test_final))
cv_scores = []

In [10]:
print(f"Début de la Cross-Validation ({n_splits} folds)...")


for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_train)):
    X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
    y_t, y_v = y_train[train_idx], y_train[val_idx]
    
    # Configuration HistGBDT
    model = HistGradientBoostingRegressor(
        max_iter=1000,
        learning_rate=0.02,
        max_leaf_nodes=31,     # Réduit pour éviter l'overfitting sur ton petit dataset
        l2_regularization=10.0, # Forte régularisation pour gérer les 512 features CLIP
        early_stopping=True,
        n_iter_no_change=50,
        random_state=42 + fold,
        verbose=0
    )
    
    model.fit(X_t, y_t)
    
    # Prédiction et Score
    val_preds = model.predict(X_v)
    fold_rmse = mean_squared_error(y_v, val_preds)
    cv_scores.append(fold_rmse)
    
    print(f"Fold {fold+1} RMSE: {fold_rmse:.4f}")
    test_preds += model.predict(X_test_final) / n_splits

Début de la Cross-Validation (5 folds)...
Fold 1 RMSE: 1.4729
Fold 2 RMSE: 1.8386
Fold 3 RMSE: 1.5847
Fold 4 RMSE: 1.5068
Fold 5 RMSE: 1.7564


In [11]:
print("-" * 30)
rmse_moyen = np.mean(cv_scores)
print(f"RMSE Moyen CV: {rmse_moyen:.4f} (+/- {np.std(cv_scores):.4f})")

# 6. Création du fichier de soumission
submission = pd.DataFrame({
    'ID': test_ids,
    'popularity': test_preds
})

submission.to_csv("submission.csv", index=False)
print("Fichier 'submission.csv' généré ! 🏆")

------------------------------
RMSE Moyen CV: 1.6319 (+/- 0.1424)
Fichier 'submission.csv' généré ! 🏆
